In [1]:
import random

def midpoint_displacement(x1, y1, x2, y2, roughness, depth):
    if depth == 0:
        return [(x1, y1), (x2, y2)]

    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2

    offset = roughness * random.uniform(-1, 1)
    mid_y = mid_y + offset

    new_r = roughness * 0.5

    left_half = midpoint_displacement(x1, y1, mid_x, mid_y, new_r, depth - 1)
    right_half = midpoint_displacement(mid_x, mid_y, x2, y2, new_r, depth - 1)

    return left_half[:-1] + right_half

In [2]:
def generate_terrain(width, height, roughness, depth, x1=0, y1=0, x2=None, y2=None, grid=None):
    if grid is None:
        grid = [[0.0 for _ in range(width + 1)] for _ in range(height + 1)]
        x2 = width
        y2 = height

        grid[0][0] = 0.0
        grid[0][x2] = 0.0
        grid[y2][0] = 0.0
        grid[y2][x2] = 0.0

    if depth == 0:
        return grid

    mid_x = (x1 + x2) // 2
    mid_y = (y1 + y2) // 2

    avg = (grid[y1][x1] + grid[y1][x2] + grid[y2][x1] + grid[y2][x2]) / 4
    grid[mid_y][mid_x] = avg + (roughness * random.uniform(-1, 1))

    grid[y1][mid_x] = (grid[y1][x1] + grid[y1][x2] + grid[mid_y][mid_x]) / 3 + (roughness * random.uniform(-1, 1))
    grid[y2][mid_x] = (grid[y2][x1] + grid[y2][x2] + grid[mid_y][mid_x]) / 3 + (roughness * random.uniform(-1, 1))
    grid[mid_y][x1] = (grid[y1][x1] + grid[y2][x1] + grid[mid_y][mid_x]) / 3 + (roughness * random.uniform(-1, 1))
    grid[mid_y][x2] = (grid[y1][x2] + grid[y2][x2] + grid[mid_y][mid_x]) / 3 + (roughness * random.uniform(-1, 1))

    new_r = roughness * 0.5

    generate_terrain(width, height, new_r, depth - 1, x1, y1, mid_x, mid_y, grid)
    generate_terrain(width, height, new_r, depth - 1, mid_x, y1, x2, mid_y, grid)
    generate_terrain(width, height, new_r, depth - 1, x1, mid_y, mid_x, y2, grid)
    generate_terrain(width, height, new_r, depth - 1, mid_x, mid_y, x2, y2, grid)

    return grid

In [3]:
def detect_artifacts(terrain_grid, threshold):
    suspects = []

    height = len(terrain_grid)
    width = len(terrain_grid[0])

    for y in range(height - 1):
        for x in range(width - 1):
            diff_x = abs(terrain_grid[y][x] - terrain_grid[y][x + 1])
            diff_y = abs(terrain_grid[y][x] - terrain_grid[y + 1][x])

            if diff_x > threshold or diff_y > threshold:
                suspects.append((y, x))

    return suspects